# Latent-I2SB sampling from a trained run

Loads a trained **latent-I2SB** run from its saved `config.json`: builds the regressor `R`,
applies its **EMA** weights, and loads the two **frozen** dictionaries (`D_joint`, `D_t1ce`)
named in `cfg["dicts"]`. Then runs the reverse Schrodinger bridge **entirely in the M-channel
latent** (`z1 -> z0`) and decodes to the T1CE image **once** at the end.

This mirrors `i2sb_sample.ipynb`, but the bridge lives in latent space, so there are three
networks instead of one. Point `CONFIG_PATH` at the latent run's saved `config.json`; the frozen
dict paths, bridge schedule (incl. the latent-tuned `tau`), and data block are all read from it.
Run on the HPC node where the data + checkpoints live. Requires steps 1 & 2 (the two dictionaries)
to have been trained first.

In [ ]:
import os, sys, json, time
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# ---- point this at your trained latent-I2SB run (the SAVED config.json) ----
CONFIG_PATH = "trained_nets/brats/Latent_I2SB_T1ce/config.json"

# ---- sampling knobs ----
SPLIT     = "val"      # which data split to sample from
USE_EMA   = True       # sample R with EMA weights (recommended)
NFE       = 100        # number of latent denoising steps (higher = better/slower; < n_points)
OT_ODE    = None       # None = use config's ot_ode; True = deterministic; False = stochastic
LOG_COUNT = 6          # trajectory frames to keep (decoded from intermediate latents)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
npy = lambda t: t.detach().cpu().numpy()
print("repo:", REPO_ROOT, "| device:", device)

## 1. Load config, build `R`, apply EMA, load the frozen dictionaries

`R` is built from `cfg["model"]` (the saved config has `init` disabled) and gets its EMA weights,
exactly like `i2sb_sample.ipynb`. The two dictionaries are loaded with the SAME helper the training
loop uses (`_load_frozen_dict` = `load_model` + freeze + eval + compile flex), from the paths in
`cfg["dicts"]`.

In [ ]:
from models import build_model
from training.latent_i2sb import _load_frozen_dict

with open(CONFIG_PATH) as f:
    cfg = json.load(f)
ic  = cfg["i2sb"]
dcfg = cfg["dicts"]

save_dir  = os.path.dirname(CONFIG_PATH)
ckpt_path = os.path.join(save_dir, "net.ckpt")     # derive from the config dir (robust)
ema_path  = os.path.join(save_dir, "ema.pt")

# --- the trained regressor R ---
R = build_model(cfg).to(device)
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
R.load_state_dict(ckpt["model_state_dict"])
R.eval()
print(f"loaded R weights from {ckpt_path} (step {ckpt.get('step')})")

def apply_ema_(net, ema_path, device):
    """Copy the saved EMA shadow into the model params (same order the EMA class used)."""
    sd = torch.load(ema_path, map_location=device, weights_only=False)
    params = [p for p in net.parameters() if p.requires_grad]
    shadow = sd["shadow"]
    assert len(params) == len(shadow), f"EMA/param mismatch: {len(shadow)} vs {len(params)}"
    for p, s in zip(params, shadow):
        p.data.copy_(s.to(p.device))

if USE_EMA and os.path.exists(ema_path):
    apply_ema_(R, ema_path, device); print(f"applied EMA weights from {ema_path}")
elif USE_EMA:
    print(f"[warn] {ema_path} not found -> sampling with raw weights")

if getattr(R, "attn_backend", None) == "flex":
    R.compile_flex()                               # parity with train.py main()

# --- the two frozen dictionaries (paths are relative to the repo root) ---
D_joint = _load_frozen_dict(dcfg["joint_dict_config"], device)
D_t1ce  = _load_frozen_dict(dcfg["t1ce_dict_config"],  device)

# invariants the whole design rests on (mirror _assert_latent_shapes)
assert D_joint.M == D_t1ce.M and D_joint.sc == D_t1ce.sc, "dicts must share the latent (M, sc)"
M           = D_joint.M
conditional = ic.get("conditional", True)
decode_dc   = ic.get("decode_dc", "x1_mean")
assert R.C == (2 * M if conditional else M), f"R.C={R.C} != {(2*M if conditional else M)}"

print(f"dicts: joint C={D_joint.C} M={D_joint.M} sc={D_joint.sc} | "
      f"t1ce C={D_t1ce.C} M={D_t1ce.M} sc={D_t1ce.sc} | R C={R.C} M={R.M}")
print(f"conditional={conditional}  decode_dc={decode_dc!r}")
print("data  : image_key =", cfg["data"][SPLIT].get("image_key", "img"),
      "| scales =", cfg["data"][SPLIT].get("scales"))

## 2. Rebuild the latent bridge from the config

Same schedule the regressor was trained on. NOTE the `std_sb` / `std_fwd` printed here are in
**latent units** (the `tau` was tuned to the code scale in `inspect_latent_tau.ipynb`), not image
units.

In [ ]:
from sb.base import build_schedule, n_steps, bridge_coeffs

bridge = build_schedule(
    kind=ic.get("kind", "brownian"),
    n_points=ic.get("n_points", 1000),
    device=device,
    tau=ic.get("tau", 0.19),
    beta_max=ic.get("beta_max", 0.3),
)
N = n_steps(bridge)
print(f"bridge_type = {ic.get('kind')}   n_points = {N}   tau = {ic.get('tau')}")
print(f"max std_fwd (R sigma ceiling, latent units) = {float(bridge.std_fwd.max()):.4f}")
print(f"max std_sb  (peak latent noise)             = {float(bridge_coeffs(bridge)[2].max()):.4f}")
assert 0 < NFE < N, f"NFE must be in (0, {N})"

## 3. Load data exactly as trained

Uses the `i2sb` loader from `cfg["data"][SPLIT]`, so `image_key`, `scales`, crop, endpoints and
`cond_idx` match training. Conditioning is REQUIRED here (the joint dictionary is conditioned on
FLAIR/T1/T2), so `cond` stays a tensor.

In [ ]:
import datasets                          # registers the loaders
from datasets.registry import build_loader

data_cfg = dict(cfg["data"][SPLIT])
data_cfg["num_workers"] = 0
loader = build_loader(data_cfg, shuffle=True, drop_last=False)

x0, x1, cond, mask = next(iter(loader))    # x0 = T1CE (GT), x1 = T1 (prior), cond = FLAIR/T1/T2
x0, x1, mask = x0.to(device), x1.to(device), mask.to(device)
cond = cond.to(device)
assert cond.shape[1] > 0, "latent-I2SB needs conditioning channels (joint dict C = 1 + n_cond)"

# contrast labels for display, derived from the stored order [flair, t1, t1ce, t2]
CONTRASTS = ["FLAIR", "T1", "T1CE", "T2"]
cond_idx = list(data_cfg.get("cond_idx", [0, 1, 3]))
cond_labels = [CONTRASTS[i] for i in cond_idx]
print("x0 (T1CE GT) :", tuple(x0.shape))
print("x1 (T1 prior):", tuple(x1.shape))
print("cond         :", tuple(cond.shape), "=", cond_labels)

## 4. Sample `z1 -> z0` in the latent domain, then decode once

Calls the library sampler `latent_i2sb_sample` (the same one the training loop's `full_recon`
validation uses): encode the prior to `z1`, run the DDPM posterior in latent space, decode the
final latent with `D_t1ce.D`. Only the final image comes back.

In [ ]:
from sb.latent_i2sb import latent_i2sb_sample

deterministic = ic.get("deterministic", False) if OT_ODE is None else OT_ODE
t0 = time.time()
recon = latent_i2sb_sample(
    D_joint, R, D_t1ce, x1, cond, bridge,
    conditional=conditional, M=M, nfe=NFE, deterministic=deterministic,
    decode_dc=decode_dc, verbose=True,
)
print()
print(f"sampled {recon.shape[0]} images in {time.time()-t0:.1f}s (nfe={NFE}, deterministic={deterministic})")
print("recon:", tuple(recon.shape))

## 5. Results:  `T1 prior | conditioning | latent-I2SB recon | GT | residual`

In [ ]:
from training.metrics import compute_metrics
from training.common import apply_loss_mask

def disp_window(*imgs, m=None, lo=1, hi=99):
    vals = [(im[m.bool().expand_as(im)] if m is not None else im).flatten() for im in imgs]
    v = torch.cat(vals).float().cpu()
    if v.numel() > 1_000_000:
        v = v[torch.randint(0, v.numel(), (1_000_000,))]
    return float(torch.quantile(v, lo / 100)), float(torch.quantile(v, hi / 100))

x0m, reconm = apply_loss_mask(x0, recon, mask, use_mask=True)
nshow = min(4, x0.shape[0])
vmin, vmax = disp_window(x1[:nshow], x0[:nshow], m=mask[:nshow])

ncols = 3 + cond.shape[1]                          # prior + each cond + recon + GT + residual
fig, ax = plt.subplots(nshow, ncols, figsize=(2.1 * ncols, 2.2 * nshow))
ax = np.atleast_2d(ax)
for i in range(nshow):
    g, r = x0m[i:i+1], reconm[i:i+1]
    mets = compute_metrics(g, r, psnr_only=False)
    panels = [(x1[i, 0], "x1 = T1 (prior)")]
    panels += [(cond[i, c], cond_labels[c]) for c in range(cond.shape[1])]
    panels += [(reconm[i, 0], f"recon  PSNR={float(mets['psnr']):.1f}"),
               (x0m[i, 0], "x0 = T1CE (GT)")]
    for a, (im, ttl) in zip(ax[i, :ncols - 1], panels):
        a.imshow(npy(im), cmap="gray", vmin=vmin, vmax=vmax); a.set_title(ttl, fontsize=8)
    rr = npy((g - r).abs()[0, 0])
    ax[i, ncols - 1].imshow(rr, cmap="magma", vmin=0, vmax=np.percentile(rr, 99))
    ax[i, ncols - 1].set_title("|GT - recon|", fontsize=8)
    for a in ax[i]: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 6. Reverse-sampling trajectory (decode intermediate latents)

`latent_i2sb_sample` only returns the final image, so here we run the same latent DDPM posterior
but log several intermediate latents and decode each one, to watch the bridge move from the prior
toward T1CE. Logic is kept identical to the library sampler (same `encode` / `latent_regress` /
`decode` / DC), just with more `log_steps`.

In [ ]:
from sb.base import space_indices
from sb.base import reverse_sample, forward_std, bridge_coeffs
from sb.latent_i2sb import encode, decode, latent_regress, _decode_dc

@torch.no_grad()
def latent_sample_with_traj(x1, cond, log_count=LOG_COUNT):
    z1 = encode(D_joint, torch.cat([x1, cond], dim=1))
    cond_lat = z1 if conditional else None
    steps = space_indices(N, NFE + 1)
    log_count = min(len(steps) - 1, log_count)
    log_steps = [steps[i] for i in space_indices(len(steps) - 1, log_count)]  # includes 0

    def pred_x0_fn(zt, step):
        step_t = torch.full((zt.shape[0],), step, device=device, dtype=torch.long)
        sig = forward_std(bridge, step_t, xdim=zt.shape[1:])
        return latent_regress(R, zt, cond_lat, sig, M)

    _, zs, _ = reverse_sample(bridge, pred_x0_fn, z1, nfe=NFE, deterministic=deterministic,
                              log_count=log_count, verbose=False)
    dc = _decode_dc(x1, decode_dc)                 # per-image; broadcasts over frames
    frames = [decode(D_t1ce, zs[:, j].to(device), dc=dc).cpu() for j in range(zs.shape[1])]
    return torch.stack(frames, dim=1)              # (B, log_count, 1, H, W); frame 0 = final T1CE

traj = latent_sample_with_traj(x1, cond)
b = min(3, traj.shape[0] - 1)
frames = traj[b]                                   # (LOG_COUNT, 1, H, W)
L = frames.shape[0]
fig, ax = plt.subplots(1, L, figsize=(2.6 * L, 2.9))
for j in range(L):
    ax[j].imshow(npy(frames[j, 0]), cmap="gray", vmin=vmin, vmax=vmax)
    ax[j].set_xticks([]); ax[j].set_yticks([]); ax[j].set_title(f"frame {j}", fontsize=9)
fig.suptitle("Latent reverse sampling: frame 0 = final T1CE  ...  last = near prior", y=1.03)
plt.tight_layout(); plt.show()

## 7. Aggregate metrics over a few val batches (optional)

Full latent sampling per batch, so keep `N_BATCHES` small. Metrics are masked, matching the
training loss.

In [ ]:
N_BATCHES = 4
agg = {"psnr": 0.0, "ssim": 0.0, "nrmse": 0.0}
n = 0
it = iter(loader)
for bi in range(N_BATCHES):
    try:
        a0, a1, ac, am = next(it)
    except StopIteration:
        break
    a0, a1, am = a0.to(device), a1.to(device), am.to(device)
    ac = ac.to(device)
    rc = latent_i2sb_sample(D_joint, R, D_t1ce, a1, ac, bridge,
                            conditional=conditional, M=M, nfe=NFE, deterministic=deterministic,
                            decode_dc=decode_dc, verbose=False)
    g, r = apply_loss_mask(a0, rc, am, use_mask=True)
    m = compute_metrics(g, r, psnr_only=False)
    bs = a0.shape[0]
    for k in agg: agg[k] += float(m[k]) * bs
    n += bs
    print(f"batch {bi+1}/{N_BATCHES}: PSNR={float(m['psnr']):.2f} "
          f"SSIM={float(m['ssim']):.3f} NRMSE={float(m['nrmse']):.4f}  (n={n})")
print()
print(f"MEAN over {n} images:", {k: round(v / max(n, 1), 4) for k, v in agg.items()})